# Скрининг риска анемии по фото глаза — v12

**Не диагноз.** Результат — шкала риска: **Норма / Лёгкая анемия / Умеренная анемия (или тяжелее)**, либо **Переснять фото** / **Оценка ненадёжна**, и гемоглобин словами.

**Что исправлено по сравнению с v11**
1. **Фото только конъюнктивы больше не отбраковывается.** В v11 поиск конъюнктивы учился только на кадрах «весь глаз» и опирался на положение относительно радужки; на крупном плане века радужки нет → маска пустая → «переснять». Теперь каждая ткань ищется по своим свойствам (`anemia/regions.py`): склера — самая светлая и бесцветная ткань у радужки, конъюнктива — розово-красная ткань под склерой (CIELAB a*, угол оттенка), край уточняет GrabCut. Уже вырезанная слизистая (как в CP-AnemiC) распознаётся отдельно.
2. **Три режима.** Поле «Что на фото» в ячейке 4: *Авто*, *Только конъюнктива*, *Только склера*, *Конъюнктива и склера*. Видны обе ткани — модель по обеим; видна одна — модель только по ней; ни одной — «Переснять».
3. **Проверка правдоподобия.** Найденная «конъюнктива» должна быть красно-розовой, «склера» — бесцветной и светлее конъюнктивы. Кожа, тень или блик не выдаются за ткань.
4. **Ложная «тяжёлая анемия».** Модели учатся на уравненных классах и без поправки завышают анемию у здоровых (на CP-AnemiC v11 называл «умеренная+» 18% здоровых детей). Теперь вероятности исправляются на реальную частоту классов, а снимок, цвета которого далеко за пределами обучающих фото (другой свет или камера), получает «Оценка ненадёжна» вместо цифры.

**Как запустить:** «Среда выполнения → Выполнить все». Первый запуск обучает модели один раз (≈3–6 мин, из них 1–2 мин скачивание Eyes-defy-anemia с Kaggle). Ячейка 3 скачает `anemia_models_v12.joblib` — в следующий раз перетащите его в «Файлы» слева, и модели загрузятся за секунды.

**Точность** печатается в ячейке 3 и рисуется в ячейке 5 — только на пациентах, которых модель не видела, включая совпадение найденной конъюнктивы с разметкой врачей (Dice).


In [ ]:
#@title ЯЧЕЙКА 1 — НАСТРОЙКА { display-mode: "form" }
#@markdown **Google Диск** — чтобы обученные модели сохранились и в следующий раз загрузились за секунды.
#@markdown Без Диска модели сохраняются файлом `anemia_models_v12.joblib`: ячейка 3 скачает его на компьютер,
#@markdown в следующий раз просто перетащите его в панель «Файлы» слева перед запуском.
Использовать_Google_Диск = False  #@param {type:"boolean"}
#@markdown Ветка репозитория, из которой берётся код поиска конъюнктивы и склеры (`anemia/regions.py`).
#@markdown После слияния PR оставьте `main`.
Ветка = 'main'  #@param {type:"string"}

import os, sys, glob, shutil, subprocess, time, warnings, urllib.request, importlib
warnings.filterwarnings('ignore')
T0 = time.time()

# Пакет anemia из репозитория (публичный): поиск тканей anemia/regions.py. --no-deps: не трогаем numpy/opencv Colab
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall',
                f'git+https://github.com/zhansaya-afk/jjghg@{Ветка.strip() or "main"}'], check=True)
# MediaPipe ставим без зависимостей: так он не трогает numpy/opencv Colab и не требует перезапуска среды
try:
    import mediapipe  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'mediapipe'], check=False)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flatbuffers', 'absl-py'], check=False)
if not os.path.exists('face_landmarker.task'):
    try:
        urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/'
                                   'face_landmarker/float16/1/face_landmarker.task', 'face_landmarker.task')
    except Exception as e:
        print('⚠️ Модель поиска лица не скачалась (фото всего лица не разобрать, крупные планы работают):', str(e)[:100])

import numpy as np, pandas as pd, cv2, sklearn, joblib
import matplotlib.pyplot as plt
import anemia.regions as regions
importlib.reload(regions)
plt.rcParams['figure.dpi'] = 100

MODEL_NAME = 'anemia_models_v12.joblib'
WORK = 'anemia_v12'
ON_DRIVE = False
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK, ON_DRIVE = '/content/drive/MyDrive/anemia_v12', True
    except Exception as e:
        print('⚠️ Google Диск не подключился, работаю без него:', str(e)[:120])
os.makedirs(WORK, exist_ok=True)
MODEL_PATH = f'{WORK}/{MODEL_NAME}'
if not os.path.exists(MODEL_PATH) and os.path.exists(MODEL_NAME):      # файл моделей перетащили в «Файлы»
    shutil.copy(MODEL_NAME, MODEL_PATH)

print(f'✅ Ячейка 1 готова за {time.time() - T0:.0f} с · OpenCV {cv2.__version__} · scikit-learn {sklearn.__version__}'
      f' · код тканей из ветки «{Ветка}»')
print('   Модели:', 'найдены — обучение не понадобится' if os.path.exists(MODEL_PATH)
      else 'нет — ячейка 3 обучит их один раз (≈3–6 мин)')


In [ ]:
#@title ЯЧЕЙКА 2 — БИБЛИОТЕКА (только определения функций) { display-mode: "form" }
# Конвейер v12:
#   фото → что на снимке (вырезанная ткань / крупный план века / глаз / лицо — MediaPipe)
#        → конъюнктива и склера по их собственным свойствам (anemia/regions.py, см. описание в начале файла)
#        → проверка правдоподобия каждой ткани → признаки цвета
#        → модель «конъюнктива», «склера» или «обе» — по тому, какие ткани реально найдены
#        → вероятности с поправкой на частоту классов → проверка «фото похоже на обучающие» → результат
import os, glob, json, time, re, warnings
import numpy as np, cv2
from anemia.regions import detect_regions, quality as tissue_quality, MODES
warnings.filterwarnings('ignore')

EPS = 1e-4
CLASSES = ['Норма', 'Лёгкая анемия', 'Умеренная анемия (или тяжелее)']
HB_WORDS = ['Гемоглобин в норме', 'Гемоглобин немного снижен', 'Гемоглобин низкий']
RETAKE = 'Переснять фото'
UNRELIABLE = 'Оценка ненадёжна'
MODE_OF = {'Авто': 'auto', 'Только конъюнктива': 'conj', 'Только склера': 'sclera', 'Конъюнктива и склера': 'both'}

# ------------------------------------------------------------------ пороги ВОЗ (г/дл)
def who_thresholds(sex, age, pregnant=False):
    """(граница нормы, граница умеренной). Ниже первой — анемия, ниже второй — умеренная и тяжелее."""
    if age is not None and age < 5:
        return 11.0, 10.0
    if pregnant:
        return 11.0, 10.0
    if age is not None and age < 12:
        return 11.5, 11.0
    if age is not None and age < 15:
        return 12.0, 11.0
    return (13.0, 11.0) if sex == 'M' else (12.0, 11.0)

def hb_class(hb, thr_norm, thr_mod):
    return 0 if hb >= thr_norm else 1 if hb >= thr_mod else 2

# ------------------------------------------------------------------ загрузка
def load_rgb(path, max_side=1600):
    """RGB float32 [0,1], с учётом поворота EXIF, поддержка HEIC."""
    from PIL import Image, ImageOps
    try:
        im = Image.open(path)
    except Exception:
        try:
            from pillow_heif import register_heif_opener
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow-heif'], check=False)
            from pillow_heif import register_heif_opener
        register_heif_opener()
        im = Image.open(path)
    im.draft('RGB', (max_side, max_side))
    im = ImageOps.exif_transpose(im).convert('RGB')
    rgb = np.asarray(im).astype(np.float32) / 255
    h, w = rgb.shape[:2]
    s = max_side / max(h, w)
    return rgb if s >= 1 else cv2.resize(rgb, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)

# ------------------------------------------------------------------ цвет и свет
def srgb_to_lin(x):
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)

def lin_to_srgb(x):
    x = np.clip(x, 0, 1)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)

def relight(rgb, rng, strength=1.0):
    """Симуляция другой лампы и камеры: баланс белого, экспозиция, гамма (для обучения и стресс-теста)."""
    lin = srgb_to_lin(np.clip(rgb, 0, 1))
    t = rng.normal(0, 0.18 * strength)
    tint = rng.normal(0, 0.06 * strength)
    gains = np.exp(np.array([t, tint, -t], np.float32))
    expo = np.exp(rng.normal(0, 0.30 * strength))
    out = lin_to_srgb(lin * gains * expo)
    return (np.clip(out, 0, 1) ** np.exp(rng.normal(0, 0.10 * strength))).astype(np.float32)

def white_balance_by(rgb, ref_mask):
    """Эталон белого — склера: делим линейный RGB на средний цвет склеры (модель фон Криса)."""
    lin = srgb_to_lin(np.clip(rgb, 0, 1))
    ref = np.median(lin[ref_mask], axis=0) + 1e-4
    lin = lin * (ref.mean() / ref)
    return lin_to_srgb(lin).astype(np.float32)

# ------------------------------------------------------------------ признаки области (как в v11: с ними посчитан cp_anemic_features.npz)
def tissue_mask(rgb, mask):
    """Убираем блики (очень светлые и бесцветные), тени и ресницы (тёмные)."""
    v = rgb.max(-1)
    s = (v - rgb.min(-1)) / (v + EPS)
    m = mask & (v > 0.08) & ~((v > 0.92) & (s < 0.25))
    if m.sum() < 30:
        return mask & (v > 0.03)
    lo, hi = np.percentile(v[m], [3, 99])
    return m & (v >= lo) & (v <= hi)

def roi_features(rgb, mask, prefix):
    m = tissue_mask(rgb, mask)
    if m.sum() < 30:
        return None
    px = np.clip(rgb[m].astype(np.float32), 1 / 255, 1)
    R, G, B = px[:, 0], px[:, 1], px[:, 2]
    lab = cv2.cvtColor(px[None], cv2.COLOR_RGB2Lab)[0]
    L, A, Bb = lab[:, 0], lab[:, 1], lab[:, 2]
    hsv = cv2.cvtColor(px[None], cv2.COLOR_RGB2HSV)[0]
    hue = np.where(hsv[:, 0] > 180, hsv[:, 0] - 360, hsv[:, 0])
    S = R + G + B
    ch = {'EI': np.log10(R) - np.log10(G),            # Erythema Index
          'lRB': np.log(R / B), 'lGB': np.log(G / B),
          'r': R / S, 'g': G / S, 'a': A, 'b': Bb, 'L': L, 'hue': hue, 'sat': hsv[:, 1],
          'C': np.hypot(A, Bb), 'hab': np.degrees(np.arctan2(Bb, A))}
    f = {}
    for k, v in ch.items():
        p10, p50, p90 = np.percentile(v, [10, 50, 90])
        f[k + '_m'], f[k + '_p10'], f[k + '_p50'], f[k + '_p90'], f[k + '_iqr'] = \
            float(v.mean()), float(p10), float(p50), float(p90), float(p90 - p10)
    ei = ch['EI']
    f['HHR'] = float(((hue > -10) & (hue < 10)).mean())        # High Hue Ratio
    f['vessel'] = float(np.percentile(ei, 95) - np.median(ei))  # контраст сосудов
    f['a_over_L'] = float(np.median(A) / (np.median(L) + EPS))
    return {prefix + k: v for k, v in f.items()}

def eye_features(rgb, conj, sclera):
    """Признаки для моделей. c_ — конъюнктива, s_ — склера, w_ — конъюнктива после баланса белого по склере."""
    out = {}
    fc = roi_features(rgb, conj, 'c_') if conj is not None and conj.sum() >= 30 else None
    fs = roi_features(rgb, sclera, 's_') if sclera is not None and sclera.sum() >= 30 else None
    if fc:
        out.update(fc)
    if fs:
        out.update(fs)
    if fc and fs:
        fw = roi_features(white_balance_by(rgb, tissue_mask(rgb, sclera)), conj, 'w_')
        if fw:
            out.update(fw)
    return out

# ------------------------------------------------------------------ вероятности и «похоже ли фото на обучающие»
def prior_correct(P, prior):
    """Модель училась на уравненных классах (каждый класс 1/3), поэтому завышает редкие (анемию).
    Байесовская поправка возвращает реальную частоту классов: P_c ∝ P_c · prior_c / (1/3)."""
    Q = np.asarray(P) * np.asarray(prior)[None] if np.ndim(P) == 2 else np.asarray(P) * np.asarray(prior)
    return Q / Q.sum(-1, keepdims=True)

def ood_share(x, lo, hi):
    """Доля признаков далеко за пределами обучающих (за 1–99 перцентилем плюс четверть размаха)."""
    span = np.maximum(hi - lo, 1e-6)
    return float(np.mean((x < lo - 0.25 * span) | (x > hi + 0.25 * span)))

OOD_MAX = 0.30          # больше 30% признаков вне обучающего диапазона — снимок «не такой», цифре не верим

# ------------------------------------------------------------------ поиск глаза на лице (MediaPipe)
_FL = None
FL_PATH = 'face_landmarker.task'

def landmarker():
    global _FL
    if _FL is None:
        try:
            from mediapipe.tasks.python import vision, BaseOptions
            opt = vision.FaceLandmarkerOptions(base_options=BaseOptions(model_asset_path=FL_PATH), num_faces=1,
                                               min_face_detection_confidence=0.3, min_face_presence_confidence=0.3)
            _FL = vision.FaceLandmarker.create_from_options(opt)
        except Exception as e:
            print('ℹ️ MediaPipe недоступен: фото лица целиком не разобрать, крупные планы работают:', str(e)[:120])
            _FL = False
    return _FL

print('✅ Ячейка 2: функции готовы')


In [ ]:
#@title ЯЧЕЙКА 2б — ДАННЫЕ CP-AnemiC (признаки цвета 251 фото) { display-mode: "form" }
# Признаки посчитаны функцией roi_features из ячейки 2 по исходным фото + 6 копиям при другом свете; Hb из таблицы датасета.
# Репозиторий публичный, поэтому медицинские данные в нём не хранятся: файл cp_anemic_features.npz
# перетащите в панель «Файлы» слева (или положите в папку моделей на Google Диске) перед запуском.
import base64
CP_FEATS_FILE = 'cp_anemic_features.npz'
_cp_path = next((p for p in (CP_FEATS_FILE, f'{WORK}/{CP_FEATS_FILE}') if os.path.exists(p)), None)
if _cp_path is None:
    try:
        from google.colab import files
        print(f'Загрузите {CP_FEATS_FILE}:')
        files.upload()
        _cp_path = CP_FEATS_FILE if os.path.exists(CP_FEATS_FILE) else None
    except Exception:
        pass
if _cp_path is None:
    raise FileNotFoundError(f'Нет {CP_FEATS_FILE}: перетащите его в «Файлы» и запустите ячейку снова')
CP_FEATS_B64 = base64.b64encode(open(_cp_path, 'rb').read()).decode()
print('✅ Ячейка 2б: признаки CP-AnemiC загружены из', _cp_path)


In [ ]:
#@title ЯЧЕЙКА 3 — ДАННЫЕ, ОБУЧЕНИЕ И ЧЕСТНАЯ ПРОВЕРКА ТОЧНОСТИ { display-mode: "form" }
# Первый запуск: скачивание Eyes-defy-anemia (Kaggle) и обучение ≈3–6 мин. Дальше — загрузка файла моделей за секунды.
# Все цифры точности — на пациентах, которых модель НЕ видела при обучении (кросс-валидация по пациентам).
import io, base64
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score

VERSION = 'v12.0'
N_RELIGHT = 3                      # копий каждого фото при другом свете

def _clf():
    return HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                          l2_regularization=1.0, min_samples_leaf=15, random_state=0)

def _fit(X, y, seed=0):
    """Классы уравниваются повторением редких примеров (вместо sample_weight: в новых scikit-learn веса
    замедляют обучение в 30 раз). Перекос этого уравнивания убирает prior_correct при ответе."""
    rng = np.random.default_rng(seed)
    cls, cnt = np.unique(y, return_counts=True)
    idx = np.concatenate([np.concatenate([np.flatnonzero(y == c),
                                          rng.choice(np.flatnonzero(y == c), cnt.max() - n, replace=True)])
                          for c, n in zip(cls, cnt)])
    return _clf().fit(X[idx], y[idx])

def proba3(m, X):
    """Вероятности трёх классов даже если какого-то класса не было в обучении."""
    P = np.zeros((len(X), 3))
    P[:, m.classes_.astype(int)] = m.predict_proba(X)
    return P

def class_prior(y):
    return (np.bincount(np.asarray(y, int), minlength=3) + 1) / (len(y) + 3)

# ------------------------------------------------------------------ Eyes-defy-anemia
def _norm_id(x):
    m = re.search(r'\d+', str(x))
    return str(int(m.group())) if m else None

def _country(p):
    p = p.lower()
    return 'India' if 'india' in p else 'Italy' if 'italy' in p else 'NA'

def _find_col(cols, pats):
    for c in cols:
        if any(re.search(p, str(c).strip().lower()) for p in pats):
            return c
    return None

def eda_index(root):
    recs = []
    for t in glob.glob(f'{root}/**/*', recursive=True):
        if not t.lower().endswith(('.xlsx', '.xls', '.csv')):
            continue
        try:
            sheets = {'csv': pd.read_csv(t)} if t.lower().endswith('.csv') else pd.read_excel(t, sheet_name=None)
        except Exception:
            continue
        for df in sheets.values():
            hb = _find_col(df.columns, [r'^hg?b', r'hgb', r'hemoglob', r'haemoglob', r'emoglob'])
            pid = _find_col(df.columns, [r'^n\.?$', r'^no\.?$', r'^number', r'^id', r'patient', r'numero']) or df.columns[0]
            sex = _find_col(df.columns, [r'sex', r'gender', r'sesso'])
            if hb is None:
                continue
            for _, r in df.iterrows():
                try:
                    hbv = float(str(r[hb]).replace(',', '.'))
                except ValueError:
                    continue
                sx = str(r[sex]).strip().lower()[:1] if sex else ''
                recs.append({'country': _country(t), 'pid': _norm_id(r[pid]), 'Hb': hbv,
                             'sex': 'M' if sx == 'm' else 'F' if sx in ('f', 'w', 'd') else None})
    tab = pd.DataFrame(recs).dropna(subset=['pid', 'Hb'])
    folders = {}
    for p in glob.glob(f'{root}/**/*', recursive=True):
        if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
            folders.setdefault(os.path.dirname(p), []).append(p)
    rows = []
    for folder, files in folders.items():
        low = {f: os.path.basename(f).lower() for f in files}
        palp = [f for f, n in low.items() if 'palpebral' in n and 'forniceal' not in n]
        both = [f for f, n in low.items() if 'palpebral' in n and 'forniceal' in n]
        orig = [f for f, n in low.items() if 'palpebral' not in n and 'forniceal' not in n]
        if (palp or both) and orig:
            rows.append({'country': _country(folder), 'pid': _norm_id(os.path.basename(folder)),
                         'segment': sorted(palp or both)[0], 'original': sorted(orig)[0]})
    imgs = pd.DataFrame(rows)
    keys = ['country', 'pid'] if (tab.country != 'NA').any() else ['pid']
    d = imgs.merge(tab.drop_duplicates(keys), on=keys, how='inner').reset_index(drop=True)
    d['key'] = d.country.astype(str) + '_' + d.pid.astype(str)
    return d

def _mask_of(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        return None
    if im.ndim == 2:
        return im > 15
    if im.shape[2] == 4:
        return im[..., 3] > 0
    return im[..., :3].max(-1) > 15

def _dice(a, b):
    return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)

def _prep_one(row, seed):
    """Фото -> ткани новым поиском (тем же, что в ячейке 4, режим «конъюнктива и склера») + Dice с маской врача
    + признаки цвета (исходные и при другом свете). Маска врача для обучения моделей НЕ используется:
    модели учатся на тех масках, которые система найдёт и на новом фото."""
    try:
        rgb = load_rgb(row['original'], 1024)
        det = detect_regions(rgb, 'both')                  # кадры Eyes-defy — крупный план глаза, лицо не ищем
        crop, conj, sclera = det['crop'], det['conj'], det['sclera']
        doc = _mask_of(row['segment'])
        dice = np.nan
        if doc is not None and abs(doc.shape[0] / doc.shape[1] - rgb.shape[0] / rgb.shape[1]) < 0.02:
            doc = cv2.resize(doc.astype(np.uint8), (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
            dice = _dice(conj, doc) if doc.sum() >= 30 else np.nan
        rows = [{'key': row['key'], 'aug': 0, **eye_features(crop, conj, sclera)}]
        rng = np.random.default_rng(seed)
        for a in range(1, N_RELIGHT + 1):
            rows.append({'key': row['key'], 'aug': a, **eye_features(relight(crop, rng), conj, sclera)})
        return {'key': row['key'], 'dice': dice, 'use': '+'.join(sorted(det['use'])) or '—', 'rows': rows}
    except Exception as e:
        return {'key': row['key'], 'error': str(e)[:100]}

# ------------------------------------------------------------------ CV одной модели
def cv_model(Fd, feats, y_of, reps=2):
    """Fd: таблица признаков (aug 0 — исходные фото, aug>0 — другой свет). Вероятности сразу с поправкой
    на частоту классов обучающих фолдов — так же, как при анализе нового фото."""
    O = Fd[Fd.aug == 0].reset_index(drop=True)
    y = O.key.map(y_of).values
    P = np.zeros((reps, len(O), 3))
    PA, YA = [], []
    for rep in range(reps):
        skf = StratifiedKFold(5, shuffle=True, random_state=rep)
        for tr, te in skf.split(O, y):
            trk, tek = set(O.key.values[tr]), set(O.key.values[te])
            T = Fd[Fd.key.isin(trk)]
            m = _fit(T[feats].values, T.key.map(y_of).values)
            prior = class_prior(y[tr])
            P[rep, te] = prior_correct(proba3(m, O[feats].values[te]), prior)
            if rep == 0:
                A = Fd[Fd.key.isin(tek) & (Fd.aug > 0)]
                PA.append(prior_correct(proba3(m, A[feats].values), prior)); YA.append(A.key.map(y_of).values)
    PA, YA = np.concatenate(PA), np.concatenate(YA)
    ba = [balanced_accuracy_score(y, P[r].argmax(1)) for r in range(reps)]
    try:
        auc = roc_auc_score(y > 0, 1 - P.mean(0)[:, 0])
        auc_l = roc_auc_score(YA > 0, 1 - PA[:, 0])
    except ValueError:
        auc = auc_l = float('nan')
    cm = confusion_matrix(y, P[0].argmax(1), labels=[0, 1, 2])
    healthy = y == 0
    return {'keys': O.key.values, 'y': y, 'P': P.mean(0), 'BA': float(np.mean(ba)), 'BA_light': float(
        balanced_accuracy_score(YA, PA.argmax(1))), 'AUC': float(auc), 'AUC_light': float(auc_l),
        'cm': cm, 'recall': (cm.diagonal() / np.maximum(cm.sum(1), 1)), 'n': len(O),
        'healthy_as_severe': float((P[0].argmax(1)[healthy] == 2).mean()) if healthy.any() else float('nan'),
        'pred_counts': np.bincount(P[0].argmax(1), minlength=3)}

def print_cv(name, r):
    rec = ' / '.join(f'{v * 100:.0f}%' for v in r['recall'])
    print(f"   {name:<34} n={r['n']:<4} верный класс (сбаланс.) {r['BA'] * 100:4.0f}%  "
          f"| при другом свете {r['BA_light'] * 100:4.0f}%  | AUC анемия/нет {r['AUC']:.2f} "
          f"(другой свет {r['AUC_light']:.2f})  | найдено по классам N/Л/У: {rec}  "
          f"| здоровых назвал «умеренная+»: {r['healthy_as_severe'] * 100:.0f}%")

def model_entry(Fd, feats, y_of):
    """Финальная модель + всё, что нужно при ответе: частота классов и диапазон признаков обучающих фото."""
    X, y = Fd[feats].values, Fd.key.map(y_of).values
    O = Fd[Fd.aug == 0]
    lo, hi = np.nanpercentile(Fd[feats].values, [1, 99], axis=0)
    return {'model': _fit(X, y), 'feats': feats, 'prior': class_prior(O.key.map(y_of).values), 'lo': lo, 'hi': hi}

# ------------------------------------------------------------------ CP-AnemiC (дети до 5 лет)
def load_cp_features():
    z = np.load(io.BytesIO(base64.b64decode(CP_FEATS_B64)), allow_pickle=False)
    df = pd.DataFrame(z['X'], columns=[str(c) for c in z['cols']])
    df['key'] = ['CP_' + str(int(i)) for i in z['idx']]
    df['aug'] = z['aug'].astype(int)
    hb = dict(zip(['CP_' + str(int(i)) for i in z['hb_idx']], z['hb']))
    return df, hb

# ------------------------------------------------------------------ обучение всего
def train_all():
    t0 = time.time()
    report = {}
    # ---- 1. дети до 5 лет (CP-AnemiC, телефон, Гана): только конъюнктива
    cp, cp_hb = load_cp_features()
    cp_y = {k: hb_class(v, 11.0, 10.0) for k, v in cp_hb.items()}
    cpf = [c for c in cp.columns if c.startswith('c_')]
    print(f'1/3 CP-AnemiC (дети до 5 лет): {len(cp_hb)} фото, классы N/Л/У = '
          f'{np.bincount(list(cp_y.values()), minlength=3).tolist()}')
    r = cv_model(cp, cpf, cp_y)
    print_cv('Дети <5, конъюнктива', r)
    report['child_conj'] = r
    models = {'child_conj': model_entry(cp, cpf, cp_y)}

    # ---- 2. взрослые (Eyes-defy-anemia): ткани новым поиском + проверка по маскам врачей
    print(f'2/3 Eyes-defy-anemia: скачивание, поиск конъюнктивы и склеры... ({time.time() - t0:.0f} с)')
    import kagglehub
    root = kagglehub.dataset_download('harshwardhanfartale/eyes-defy-anemia')
    d = eda_index(root)
    if len(d) == 0:
        raise RuntimeError('В скачанном Eyes-defy-anemia не найдены фото с разметкой и Hb')
    items = Parallel(n_jobs=-1)(delayed(_prep_one)(r, i) for i, r in enumerate(d.to_dict('records')))
    bad = [it for it in items if 'error' in it]
    items = [it for it in items if 'error' not in it]
    if bad:
        print(f'   ⚠️ {len(bad)} фото не прочитались, пример: {bad[0]["error"]}')
    d = d[d.key.isin({it['key'] for it in items})].reset_index(drop=True)
    sex, hbv = dict(zip(d.key, d.sex)), dict(zip(d.key, d.Hb))
    ad_y = {k: hb_class(hbv[k], *who_thresholds(sex[k], 30)) for k in d.key}
    dice = np.array([it['dice'] for it in items], float)
    uses = pd.Series([it['use'] for it in items]).value_counts().to_dict()
    print(f'   {len(items)} пациентов, классы N/Л/У = {np.bincount(list(ad_y.values()), minlength=3).tolist()}')
    print(f'   Найдено тканей: {uses}')
    print(f'   Конъюнктива против разметки врачей (Dice): медиана {np.nanmedian(dice) * 100:.0f}%, '
          f'у четверти худших ниже {np.nanpercentile(dice, 25) * 100:.0f}%  ({time.time() - t0:.0f} с)')
    report['seg'] = {'dice': dice, 'uses': uses}

    # ---- 3. модели: по конъюнктиве, по склере, по обеим
    print(f'3/3 Признаки цвета и модели... ({time.time() - t0:.0f} с)')
    Fd = pd.DataFrame([r for it in items for r in it['rows']])
    Fd['male'] = Fd.key.map(lambda k: 1.0 if sex[k] == 'M' else 0.0 if sex[k] == 'F' else 0.5)
    sets = {'conj': [c for c in Fd.columns if c.startswith('c_')] + ['male'],
            'sclera': [c for c in Fd.columns if c.startswith('s_')] + ['male']}
    sets['both'] = sets['conj'][:-1] + sets['sclera'][:-1] + [c for c in Fd.columns if c.startswith('w_')] + ['male']
    names = {'conj': 'Взрослые, только конъюнктива', 'sclera': 'Взрослые, только склера',
             'both': 'Взрослые, конъюнктива + склера', 'child_conj': 'Дети <5 (CP-AnemiC), конъюнктива'}
    for s, feats in sets.items():
        if not all(c in Fd.columns for c in feats) or len(feats) < 3:
            print(f'   {names[s]}: ткань не найдена ни на одном фото, модель не обучается')
            continue
        sub = Fd.dropna(subset=feats)
        if sub.key.nunique() < 40:
            print(f'   {names[s]}: мало фото ({sub.key.nunique()}), модель не обучается')
            continue
        r = cv_model(sub, feats, ad_y)
        print_cv(names[s], r)
        report[s] = r
        models[s] = model_entry(sub, feats, ad_y)
    bundle = {'version': VERSION, 'sklearn': sklearn.__version__, 'models': models, 'names': names,
              'report': {k: {kk: vv for kk, vv in v.items() if kk not in ('keys', 'P')} for k, v in report.items()},
              'trained_s': time.time() - t0}
    print(f'\n✅ Обучение завершено за {time.time() - t0:.0f} с')
    return bundle

# ------------------------------------------------------------------ загрузить или обучить
T0 = time.time()
BUNDLE = None
if os.path.exists(MODEL_PATH):
    try:
        b = joblib.load(MODEL_PATH)
        if b.get('version') == VERSION and b.get('sklearn') == sklearn.__version__:
            BUNDLE = b
        else:
            print('ℹ️ Файл моделей от другой версии — обучаю заново.')
    except Exception as e:
        print('ℹ️ Файл моделей не читается — обучаю заново:', str(e)[:100])
if BUNDLE is None:
    BUNDLE = train_all()
    joblib.dump(BUNDLE, MODEL_PATH, compress=3)
    if not ON_DRIVE:
        try:
            from google.colab import files
            files.download(MODEL_PATH)
            print(f'💾 {MODEL_NAME} скачивается на компьютер: в следующий раз перетащите его в «Файлы» — обучение не понадобится.')
        except Exception:
            pass
else:
    print(f'✅ Модели загружены за {time.time() - T0:.1f} с (обучены ранее за {BUNDLE["trained_s"]:.0f} с)')


## 📷 Анализ фото
Выберите в ячейке 4 «Что на фото» (или оставьте «Авто»), пол и возраст, запустите. Снимайте при дневном свете, без вспышки.

In [ ]:
#@title 📷 ЯЧЕЙКА 4 — АНАЛИЗ ФОТО (заполните поля и запустите) { display-mode: "form" }
#@markdown **Что на фото.** «Авто» — система определит сама. Если знаете точно — выберите, так надёжнее:
#@markdown * «Только конъюнктива» — крупный план вывернутого нижнего века (или уже вырезанная слизистая);
#@markdown * «Только склера» — белок глаза, веко не оттянуто;
#@markdown * «Конъюнктива и склера» — глаз с оттянутым нижним веком: учитываются обе ткани.
Что_на_фото = 'Авто'  #@param ["Авто", "Только конъюнктива", "Только склера", "Конъюнктива и склера"]
Пол = 'Женский'  #@param ["Женский", "Мужской"]
Возраст = 20  #@param {type:"integer"}
Беременность = False  #@param {type:"boolean"}
#@markdown *(Необязательно)* путь к фото уже в Colab, например `/content/eye.jpg`. Пусто — откроется окно выбора файлов.
Путь_к_фото = ''  #@param {type:"string"}

METHOD_OF = {frozenset({'conj', 'sclera'}): 'both', frozenset({'conj'}): 'conj', frozenset({'sclera'}): 'sclera'}

def analyze_photo(path, sex, age, mode):
    t0 = time.time()
    rgb = load_rgb(path, 1600)
    det = detect_regions(rgb, mode, landmarker=landmarker())
    crop, conj, sclera = det['crop'], det['conj'], det['sclera']
    issues = tissue_quality(crop, conj, sclera, det['src_side'])
    notes = list(det['notes'])
    child = age is not None and age < 5
    method = None if issues else METHOD_OF.get(frozenset(det['use']))
    if child and method is not None:
        if 'conj' in det['use']:
            method = 'child_conj'
        else:
            method, _ = None, notes.append('для детей до 5 лет есть только модель по конъюнктиве — снимите оттянутое веко')
    P, ood, status = None, None, RETAKE
    if method is not None and method in BUNDLE['models']:
        e = BUNDLE['models'][method]
        feats = eye_features(crop, conj if 'conj' in det['use'] else None, sclera if 'sclera' in det['use'] else None)
        feats['male'] = 1.0 if sex == 'M' else 0.0
        if all(c in feats and np.isfinite(feats[c]) for c in e['feats']):
            x = np.array([feats[c] for c in e['feats']], float)
            ood = ood_share(x, e['lo'], e['hi'])
            if ood > OOD_MAX:
                status = UNRELIABLE
                notes.append(f'{ood * 100:.0f}% признаков цвета вне диапазона обучающих фото — другой свет или камера; '
                             'цифре не верим')
            else:
                P = prior_correct(proba3(e['model'], x[None])[0], e['prior'])
                status = 'ok'
        else:
            notes.append('мало чистой ткани для признаков цвета')
    elif method is not None:
        notes.append(f'модель «{method}» не обучена (мало данных)')
    return {'path': path, 'det': det, 'crop': crop, 'conj': conj, 'sclera': sclera, 'issues': issues,
            'notes': notes, 'method': method, 'P': P, 'ood': ood, 'status': status, 'seconds': time.time() - t0}

def method_reason(r):
    m = r['method']
    return {'both': 'видны конъюнктива и склера — учтены обе (цвет века дополнительно нормирован по белку глаза)',
            'conj': 'учтена только конъюнктива' + (' (склера не найдена)' if r['det']['kind'] != 'вырезанная ткань'
                                                    and 'sclera' not in r['det']['use'] else ''),
            'sclera': 'учтена только склера (конъюнктива не найдена или не запрошена) — по склере точность ниже',
            'child_conj': 'ребёнок до 5 лет — модель CP-AnemiC по конъюнктиве'}.get(m, '')

def show(results, sex, age, pregnant):
    ok = [r for r in results if r['P'] is not None]
    n = len(results)
    fig, ax = plt.subplots(1, n + 1, figsize=(4.2 * (n + 1), 4))
    ax = np.atleast_1d(ax)
    for a, r in zip(ax, results):
        vis = (np.clip(r['crop'], 0, 1) * 255).astype(np.uint8).copy()
        for m, col in ((r['conj'], (0, 255, 0)), (r['sclera'], (0, 160, 255))):
            cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(vis, cnts, -1, col, 2)
        a.imshow(vis); a.axis('off')
        a.set_title(f"{os.path.basename(r['path'])[:22]} · {r['det']['kind']}\n"
                    'зелёный — конъюнктива, голубой — склера', fontsize=9)
    out = {'version': VERSION, 'sex': sex, 'age': age, 'pregnant': pregnant, 'photos': []}
    for r in results:
        out['photos'].append({'file': os.path.basename(r['path']), 'kind': r['det']['kind'],
                              'tissues': sorted(r['det']['use']), 'method': r['method'], 'status': r['status'],
                              'issues': r['issues'], 'notes': r['notes'], 'seconds': round(r['seconds'], 2),
                              'probabilities': None if r['P'] is None else [round(float(v), 3) for v in r['P']]})
    if ok:
        P = np.mean([r['P'] for r in ok], axis=0)
        k = int(P.argmax())
        ax[-1].barh(CLASSES[::-1], P[::-1] * 100, color=['#c0392b', '#e67e22', '#27ae60'])
        ax[-1].set_xlim(0, 100); ax[-1].set_xlabel('вероятность, %'); ax[-1].set_title('Шкала риска')
        out.update({'result': CLASSES[k], 'hb': HB_WORDS[k], 'probabilities': dict(zip(CLASSES, map(float, P)))})
    else:
        ax[-1].axis('off')
        res = UNRELIABLE if any(r['status'] == UNRELIABLE for r in results) else RETAKE
        out.update({'result': res, 'hb': None})
    plt.tight_layout(); plt.show()

    print('=' * 78)
    if ok:
        conf = P.max()
        print(f"  РЕЗУЛЬТАТ: {out['result'].upper()}   ·   {out['hb']}")
        print('  ' + '   '.join(f'{c}: {p * 100:.0f}%' for c, p in zip(CLASSES, P)))
        print(f'  Риск анемии (любой степени): {(1 - P[0]) * 100:.0f}%')
        if conf < 0.5:
            second = CLASSES[int(np.argsort(P)[-2])]
            print(f'  ⚠️ Уверенность низкая ({conf * 100:.0f}%): результат между «{out["result"]}» и «{second}». '
                  'Переснимите при дневном свете или сдайте анализ крови.')
        if pregnant or (age is not None and 5 <= age < 15):
            print('  ℹ️ Модель обучена на взрослых (небеременных). Для беременных и детей 5–14 лет граница нормы по ВОЗ '
                  f'другая: {who_thresholds(sex, age, pregnant)[0]:.1f} г/дл — сверяйте с анализом крови.')
        for r in ok:
            rep = BUNDLE['report'].get(r['method'], {})
            print(f"  Ткань ({os.path.basename(r['path'])[:20]}): {method_reason(r)}. Проверенная точность метода: "
                  f"верный класс {rep.get('BA', np.nan) * 100:.0f}%, AUC {rep.get('AUC', np.nan):.2f}")
    else:
        print(f"  РЕЗУЛЬТАТ: {out['result'].upper()}")
    for r in results:
        why = r['issues'] + r['notes']
        if why:
            print(f"  📷 {os.path.basename(r['path'])[:24]}: " + '; '.join(why))
    if not ok:
        print('  Совет: дневной свет у окна, без вспышки и фильтров, глаз в фокусе; для конъюнктивы — нижнее веко '
              'оттянуто пальцем. Если знаете, что на фото, выберите это в поле «Что на фото».')
    print(f"  Время анализа: {sum(r['seconds'] for r in results):.1f} с на {n} фото")
    print('  ⚠️ Это скрининг, а не диагноз. Точный уровень гемоглобина показывает только анализ крови.')
    print('=' * 78)
    os.makedirs('results', exist_ok=True)
    with open('results/result.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    return out

sex = 'M' if Пол == 'Мужской' else 'F'
mode = MODE_OF[Что_на_фото]
paths = [p.strip() for p in Путь_к_фото.split(',') if p.strip()]
if not paths:
    from google.colab import files
    print('Выберите 1–3 фото:')
    paths = list(files.upload().keys())
results = [analyze_photo(p, sex, Возраст, mode) for p in paths[:3]]
RESULT = show(results, sex, Возраст, Беременность)


In [ ]:
#@title ЯЧЕЙКА 5 — (для защиты) ТОЧНОСТЬ: матрицы ошибок на пациентах, которых модель не видела { display-mode: "form" }
rep = BUNDLE['report']
names = BUNDLE['names']
keys = [k for k in ('both', 'conj', 'sclera', 'child_conj') if k in rep]
fig, ax = plt.subplots(1, len(keys) + 1, figsize=(4.3 * (len(keys) + 1), 4))
short = ['Норма', 'Лёгкая', 'Умер.+']
for a, k in zip(ax, keys):
    cm = rep[k]['cm']
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    a.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            a.text(j, i, f'{cm[i, j]}\n{cmn[i, j] * 100:.0f}%', ha='center', va='center',
                   color='white' if cmn[i, j] > 0.6 else 'black', fontsize=9)
    a.set_xticks(range(3)); a.set_xticklabels(short); a.set_yticks(range(3)); a.set_yticklabels(short)
    a.set_xlabel('ответ системы'); a.set_ylabel('по анализу крови')
    a.set_title(f"{names[k]}\nверный класс {rep[k]['BA'] * 100:.0f}% · AUC {rep[k]['AUC']:.2f}", fontsize=9)
if 'seg' in rep:
    dice = rep['seg']['dice'][np.isfinite(rep['seg']['dice'])]
    ax[-1].hist(dice * 100, bins=20, color='#2e86c1')
    ax[-1].set_title(f"Поиск конъюнктивы: совпадение с врачами\nмедиана {np.median(dice) * 100:.0f}%", fontsize=9)
    ax[-1].set_xlabel('Dice, %')
plt.tight_layout(); plt.show()
print('Как читать: строки — настоящий класс по анализу крови, столбцы — ответ системы. Диагональ — верные ответы.')
print('Вероятности исправлены на реальную частоту классов (prior_correct): модель училась на уравненных классах')
print('и без поправки завышала анемию у здоровых. «Здоровых назвал умеренная+» в ячейке 3 — именно эта ошибка.')
for k in keys:
    print(f"  {names[k]:<36} предсказано по классам N/Л/У: {[int(v) for v in rep[k]['pred_counts']]}, "
          f"на самом деле: {[int(v) for v in np.bincount(rep[k]['y'], minlength=3)]}")
